In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

Mounted at /content/drive
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [6]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"

!pip install arch

Agent pid 25142
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-550beff
# github.com:22 SSH-2.0-cb24e083
# github.com:22 SSH-2.0-550beff
# github.com:22 SSH-2.0-550beff
# github.com:22 SSH-2.0-550beff
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras import mixed_precision

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import precision_score, recall_score, f1_score

from functools import cached_property
from typing import Union, Optional, Dict
from scipy.stats import t as student_t



try:
    from arch import arch_model
    HAS_ARCH = True
except Exception:
    HAS_ARCH = False


#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)

#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)

# （建議）再開啟 XLA 與混合精度
tf.config.optimizer.set_jit(True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("是否可用GPU:", tf.test.is_gpu_available())
print("使用中的裝置:", tf.config.list_physical_devices('GPU'))

print("✓ GPU 初始化流程完成")


# 1. 定義檔案路徑
file_paths = {
    # "bonds_day": "./filtered_output/bonds_day_clean_period.csv",
    # "bonds_hour": "./filtered_output/bonds_hour_clean_period.csv",
    # "crypto_day": "./filtered_output/crypto_day_clean_period.csv",
    # "crypto_hour": "./filtered_output/crypto_hour_clean_period.csv",
    # "others_day": "./filtered_output/others_day_clean_period.csv",
    # "others_hour": "./filtered_output/others_hour_clean_period.csv",
    "stock_day":  "./filtered_output/stock_day_fluctuation_aligned.csv",
    # "stock_hour": "./filtered_output/stock_hour_clean_period.csv"
}


def find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df



raw_dfs = {}
for name, path in file_paths.items():
    raw_dfs[name] = read_and_clean(path)
    # print(raw_dfs[name].columns)

# ===== 資料結構：取代 all_data =====
class AssetGroupLite:
    def __init__(self, name, df):
        self.name = name
        df = df.copy()
        df['DATE'] = pd.to_datetime(df['DATE'])
        self.raw = df.set_index('DATE').sort_index()

    @cached_property
    def _close_cols(self):
        return [c for c in self.raw.columns if c.endswith('_CLOSE')]

    @cached_property
    def _vol_cols(self):
        return [c for c in self.raw.columns if c.endswith('_VOLUME')]

    @cached_property
    def close_ln(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return np.log(self.raw[self._close_cols]).rename(
            columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln')
        )

    @cached_property
    def close_ln_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        logp = np.log(self.raw[self._close_cols])
        return (
            logp.diff()
               .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln_ret'))
               .dropna(how='all')
        )

    @cached_property
    def close_arith_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return (
            self.raw[self._close_cols].pct_change()
                .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_arith_ret'))
                .dropna(how='all')
        )

class DataRepository:
    REQUIRED_INDEX = "DATE"

    def __init__(self, raw_dfs: dict, check_schema: bool = True):
        self.groups = {}
        for name, df in raw_dfs.items():
            if check_schema:
                assert 'DATE' in df.columns, f"{name}: 缺少 DATE 欄"
            self.groups[name] = AssetGroupLite(name, df)

    # 補上 group()，方便外部與內部呼叫
    def group(self, name: str) -> AssetGroupLite:
        if name not in self.groups:
            raise KeyError(f"Group '{name}' 不存在。可用群組：{list(self.groups.keys())}")
        return self.groups[name]

    def series(self, group: str, series_name: str) -> pd.Series:
        g = self.group(group)
        # 加上 raw → 能抓 OHLCV、IS_TRADING 等原始欄
        search_order = ['close_ln_ret', 'close_arith_ret', 'close_ln', 'raw']

        # 1) 直接命中
        for key in search_order:
            tbl = getattr(g, key)
            if series_name in tbl.columns:
                return tbl[series_name]

        # 2) 容錯：只給 base symbol，自動補候選
        base = (series_name
                .replace('_CLOSE', '')
                .replace('_OPEN', '')
                .replace('_HIGH', '')
                .replace('_LOW', '')
                .replace('_VOLUME', '')
                .replace('_IS_TRADING', '')
                .replace('_CLOSE_ln_ret', '')
                .replace('_CLOSE_arith_ret', '')
                .replace('_CLOSE_ln', ''))
        candidates = [
            f'{base}_CLOSE_ln_ret',
            f'{base}_CLOSE_arith_ret',
            f'{base}_CLOSE_ln',
            f'{base}_OPEN',
            f'{base}_HIGH',
            f'{base}_LOW',
            f'{base}_CLOSE',
            f'{base}_VOLUME',
            f'{base}_IS_TRADING'
        ]
        for cand in candidates:
            for key in search_order:
                tbl = getattr(g, key)
                if cand in tbl.columns:
                    return tbl[cand]

        raise KeyError(f"{group}: 找不到 {series_name} 或候選 {candidates}")

    # 取整張表
    def table(self, group: str, table_name: str) -> pd.DataFrame:
        g = self.group(group)
        if not hasattr(g, table_name):
            raise KeyError(f"{group}: 無表 '{table_name}'。可用表：['close_ln_ret','close_arith_ret','close_ln']")
        return getattr(g, table_name)

    # 若要直接拿 raw 的原始價/量欄位（例如 *_CLOSE 或 *_VOLUME）
    def raw_series(self, group: str, raw_col: str) -> pd.Series:
        g = self.group(group)
        if raw_col not in g.raw.columns:
            raise KeyError(f"{group}: raw 中沒有欄位 {raw_col}")
        return g.raw[raw_col]

repo = DataRepository(raw_dfs)

####################################
#########     LSTM     #############
####################################
# ------------------ Features ------------------
def build_feature_df(repo: DataRepository, group: str, symbol: str) -> pd.DataFrame:
    s_open  = repo.raw_series(group, f'{symbol}_OPEN').asfreq('D')
    s_high  = repo.raw_series(group, f'{symbol}_HIGH').asfreq('D')
    s_low   = repo.raw_series(group, f'{symbol}_LOW').asfreq('D')
    s_close = repo.raw_series(group, f'{symbol}_CLOSE').asfreq('D')
    s_vol   = repo.raw_series(group, f'{symbol}_VOLUME').asfreq('D')
    s_flag = repo.raw_series(group, f'{symbol}_IS_TRADING').asfreq('D')
    s_lnrt  = repo.series(group, f'{symbol}_CLOSE_ln_ret').asfreq('D')
    s_ln  = repo.series(group, f'{symbol}_CLOSE_ln').asfreq('D')
    df = pd.concat([
        s_lnrt.rename(f'{symbol}_LN_RET'),
        s_open.rename(f'{symbol}_OPEN'),
        s_high.rename(f'{symbol}_HIGH'),
        s_low.rename(f'{symbol}_LOW'),
        s_close.rename(f'{symbol}_CLOSE'),
        s_vol.rename(f'{symbol}_VOLUME'),
        s_flag.rename(f'{symbol}_IS_TRADING'),
        s_ln.rename(f'{symbol}_CLOSE_LN')
        ], axis=1)
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.dropna(how='any')


# ------------------ Model ------------------
###############  LSTM
def build_small_lstm(input_len: int, n_features: int) -> tf.keras.Model:
    inp = layers.Input(shape=(input_len, n_features))
    x = layers.LSTM(LSTM_UNITS, return_sequences=False)(inp)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(1, activation='linear')(x)
    m = models.Model(inp, out)
    # loss_fn = 'mse'
    m.compile(optimizer=optimizers.Adam(learning_rate=LR), loss='mse')  # ← 改用 MSE（或 Huber(delta≈0.05)）
    return m

############### mcHARCH
def fit_vol_per_window(ret_window, mode='garch'):
    y = ret_window.dropna().astype(float)
    if len(y) < 30:
        lam = 0.94
        ewma_var = y.pow(2).ewm(alpha=1-lam, adjust=False).mean()
        log_sigma_series = 0.5 * np.log(np.maximum(ewma_var.values, 1e-12))
        log_sigma_series = pd.Series(log_sigma_series, index=ewma_var.index)\
                              .reindex(ret_window.index).ffill().bfill()
        sigma_next = np.sqrt(lam * ewma_var.iloc[-1] + (1-lam) * y.iloc[-1]**2)
        return log_sigma_series, float(sigma_next)

    vol = 'HARCH' if mode.lower() == 'harch' else 'GARCH'
    p, q = (3, 0) if vol == 'HARCH' else (1, 1)

    scale = 100.0
    am = arch_model(y.values * scale, mean='Zero', vol=vol, p=p, q=q, dist='t')
    res = am.fit(disp='off')

    # 視窗內「過濾」波動：先除回 scale，再做下限保護
    sigma_series = res.conditional_volatility          # ndarray
    sigma_series = np.maximum(sigma_series / scale, 1e-12)
    log_sigma_series = np.log(sigma_series)
    log_sigma_series = pd.Series(log_sigma_series, index=y.index)\
                          .reindex(ret_window.index).ffill().bfill()

    # 一步前瞻：variance → sqrt → 除回 scale
    fvar = res.forecast(horizon=1, reindex=False).variance.values[-1, 0]
    sigma_next = float(np.sqrt(fvar) / scale)

    return log_sigma_series, sigma_next

################## 風險
def compute_var_es(
    mu: Union[float, np.ndarray, pd.Series],
    sigma: Union[float, np.ndarray, pd.Series],
    alpha: float = 0.01,
    nu: float = 6.0,
    price_base: Optional[Union[float, np.ndarray, pd.Series]] = None,
    safe_eps: float = 1e-12,
) -> Union[Dict[str, np.ndarray], pd.DataFrame]:
    """
    用 Student-t(ν) 的條件分佈，把 (μ, σ) 轉成 VaR/ES（報酬層級），
    並可選擇映射為價格層級（以 price_base = P_t 為基準）。

    參數
    ----
    mu : 預測條件均值（下一期對數報酬）  \hat{r}_{t+1}
    sigma : 預測條件標準差（下一期波動） \hat{\sigma}_{t+1}
    alpha : 左尾機率（如 0.01 或 0.05）
    nu : Student-t 自由度（建議 4~10；後續可改為估計值）
    price_base : 若提供，回傳 VaR/ES 的「價格層級」
    safe_eps : 防止數值問題的小常數

    回傳
    ----
    若輸入是 pandas.Series，回傳 DataFrame（index 對齊）；
    否則回傳 dict，鍵包含：
      - 'VaR_ret', 'ES_ret' （報酬層級）
      - 'VaR_price', 'ES_price' （若提供 price_base）
    """
    # 轉為 ndarray，並做基本數值清理
    def to_np(x):
        if isinstance(x, pd.Series):
            return x.values
        return np.asarray(x)

    mu_np    = to_np(mu)
    sigma_np = np.maximum(to_np(sigma), safe_eps)  # σ >= eps
    # NaN/無窮處理
    mu_np    = np.where(np.isfinite(mu_np),    mu_np,    0.0)
    sigma_np = np.where(np.isfinite(sigma_np), sigma_np, safe_eps)

    # t 分位點與 pdf
    q  = student_t.ppf(alpha, df=nu)                 # 左尾分位（負值）
    fq = student_t.pdf(q,  df=nu)

    # VaR / ES（報酬層級）
    var_ret = mu_np + q * sigma_np
    es_ret  = mu_np + sigma_np * ((nu + q*q)/((nu - 1.0)*(1.0 - alpha))) * fq

    out_price = {}
    if price_base is not None:
        pb = to_np(price_base)
        pb = np.where(np.isfinite(pb), pb, np.nan)
        var_price = pb * np.exp(var_ret)   # VaR 價格下界（最壞α分位）
        es_price  = pb * np.exp(es_ret)    # ES 價格（落入左尾時的期望）
        out_price = {'VaR_price': var_price, 'ES_price': es_price}

    # 決定回傳型別
    if isinstance(mu, pd.Series):
        idx = mu.index
        cols = {'VaR_ret': var_ret, 'ES_ret': es_ret}
        cols.update(out_price)
        return pd.DataFrame(cols, index=idx)
    else:
        out = {'VaR_ret': var_ret, 'ES_ret': es_ret}
        out.update(out_price)
        return out



# ------------------ Config ------------------
ASSET_SYMBOL_ES1 = 'ES1'
ASSET_SYMBOL_VIX = 'VIX'
GROUP_DAY   = 'stock_day'
TARGET_START_STR = '2005-05-31'
TARGET_END_STR   = '2007-05-31'

WINDOW_DAY = 20  # 14 days
GARCH_WINDOW_DAY = 250
MODE = 'warm'           # 'warm' or 'refit'
EPOCHS_INIT = 8
EPOCHS_STEP = 1
BATCH_SIZE = 32
LR = 5e-4
LSTM_UNITS = 64
DROPOUT = 0.2
USE_HUBER = True
NORMALIZE_FEATURES = True  # z-score inputs per window

# ------------------ Features ------------------
feature_names = ['ES1_LN_RET','ES1_OPEN','ES1_HIGH','ES1_LOW','ES1_CLOSE','ES1_VOLUME','ES1_CLOSE_LN','VIX_CLOSE_LN']

feat_ES1 = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_ES1)
feat_VIX = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_VIX)

df_day = feat_ES1.join(feat_VIX, how='left')

# --- 讓索引成為 datetime（很重要） ---
df_day.index = pd.to_datetime(df_day.index, errors='coerce')
df_day = df_day.sort_index()
df_day = df_day.loc[df_day['ES1_IS_TRADING'] == 1]

# print(df_day.head())
# print(df_day.columns)

df_day.to_csv("./filtered_output/df_day.csv", index=True)


PRED_START = pd.to_datetime(TARGET_START_STR)
PRED_END   = pd.to_datetime(TARGET_END_STR)

# 把回測期限制在資料範圍內
data_start = df_day.index.min()
data_end   = df_day.index.max()
if PRED_START < data_start: PRED_START = data_start
if PRED_END   > data_end:   PRED_END   = data_end

# 若 PRED_START 不是可用交易日，推到 >= PRED_START 的第一個交易日
try:
    PRED_START = df_day.index[df_day.index.searchsorted(PRED_START)]
except Exception:
    # 若整段都沒資料，直接報錯
    raise RuntimeError("資料期間與回測期間沒有交集，請調整 TARGET_START/END。")


first_needed = PRED_START - pd.Timedelta(days=WINDOW_DAY)
es1_min = df_day.index.min()
ES1_close_series = df_day['ES1_CLOSE']
if es1_min > first_needed:
    raise RuntimeError(f"Insufficient history: need <= {first_needed}, have from {ES1_close_series.index.min()}")



# ------------------ Model ------------------

# 1) 準備母表（確保排序與期間）
# 全歷史的交易日索引（只留 ES1 開市）
dates_all = df_day.index
# print(dates_all)

# 回測區間（仍然要取，決定哪些 t 需要預測）
mask_period = (df_day.index >= PRED_START) & (df_day.index <= PRED_END)
dates_period = df_day.index[mask_period]


# 2) 方便取用的短名
feat_df = df_day  # 與舊程式一致
model: Optional[tf.keras.Model] = None
rows = []


# 3) 走索引的滑窗，不假設日期連續
#   第 i 筆要預測的是 dates[i]（以 t 表示），窗口是 dates[i-WINDOW_DAY : i)（只到 t-1）

for t in dates_period:
    # 1) 找出 t 在「全歷史」中的**位置**（不是在 dates_period 中的 i）
    pos = dates_all.searchsorted(t)   # 或：pos = df_day.index.get_loc(t)

    # 2) 用「位置」切出兩個視窗（用 take 或 iloc，千萬不要用 dates_all[a:b]）
    if pos < max(WINDOW_DAY, GARCH_WINDOW_DAY):
      continue  # 歷史不足就跳過

    win_idx   = np.arange(pos - WINDOW_DAY, pos)
    garch_idx = np.arange(pos - GARCH_WINDOW_DAY, pos)

    win_dates   = dates_all.take(win_idx)
    garch_dates = dates_all.take(garch_idx)

    # 目標日與 t-1
    t_minus_1 = dates_all[pos - 1]   # 直接往前一格

    print(f"start {t} ")
    # print(f"start {t} | pos {pos} |　t_minus_1 {t_minus_1}")

    # print(f"garch head={list(garch_dates[:5].date)}")
    # print(f"garch tail={list(garch_dates[-5:].date)}\n")
    # print(f"Xw_period head={list(win_dates[:5].date)}")
    # print(f"Xw_period tail={list(win_dates[-5:].date)}\n")

    # --- 視窗特徵 ---
    Xw = feat_df.loc[win_dates, feature_names].copy()
    if len(Xw) != WINDOW_DAY or Xw.isna().any().any():
        # 視窗內若有缺值，略過這天
        continue

    # --- GARCH（視窗內，只用到 t-1 的 return） ---
    # 你的 feature_names 中若名稱是 ES1_LN_RET 就用它；否則改用你實際欄名
    # garch_dates有300天做完garch後便做zcore存到log_sigma300，再將其取所需時間段出來
    ret_col = 'ES1_LN_RET' if 'ES1_LN_RET' in feat_df.columns else 'LN_RET'
    log_sigma_series, sigma_next = fit_vol_per_window(
        ret_window=feat_df.loc[garch_dates, ret_col],
        mode='garch'  # 'harch' 也可
    )
    # print(len(log_sigma_series))

    mu300 = float(log_sigma_series.mean())
    sd300 = float(log_sigma_series.std(ddof=0))
    log_sigma300 = (log_sigma_series - mu300) / sd300
    Xw['LOG_SIGMA'] = log_sigma300.reindex(win_dates).ffill().fillna(0.0).values


    # --- 每窗 z-score（避免洩漏）---
    if NORMALIZE_FEATURES:
        # 排除已在 z 範圍內的報酬欄（可保留或一起 z-score，影響不大）
        cols_to_norm = ['ES1_OPEN','ES1_HIGH','ES1_LOW','ES1_CLOSE','ES1_VOLUME','ES1_CLOSE_LN','VIX_CLOSE_LN']
        mu = Xw[cols_to_norm].mean(axis=0)
        sd = Xw[cols_to_norm].std(axis=0).replace(0.0, np.nan)
        Xw[cols_to_norm] = (Xw[cols_to_norm] - mu) / sd

        Xw = Xw.fillna(0.0)

    names_this = list(Xw.columns)
    N_FEATURES_THIS = len(names_this)
    X_t = Xw[names_this].values.reshape(1, WINDOW_DAY, N_FEATURES_THIS).astype(np.float32)

    # --- 目標：下一日「標準化 ln 價」---
    ln_tm1 = float(feat_df.loc[t_minus_1, 'ES1_CLOSE_LN'])
    ln_t   = float(feat_df.loc[t,'ES1_CLOSE_LN'])
    # 目標標準化所需的均值/標準差（用視窗內 ln 價）
    mu_ln  = float(feat_df.loc[win_dates, 'ES1_CLOSE_LN'].mean())
    sig_ln = float(feat_df.loc[win_dates, 'ES1_CLOSE_LN'].std(ddof=0) + 1e-12)
    y_z_t  = np.array([(ln_t - mu_ln)/sig_ln], dtype=np.float32)

    # --- 先 predict 再 fit（walk-forward）---
    if model is not None:
        z_hat      = float(model.predict(X_t, verbose=0).ravel()[0])   # 預測的 z(lnP)
        yhat_ln_t  = z_hat * sig_ln + mu_ln                            # 反標準化成 lnP
        price_pred = float(np.exp(yhat_ln_t))
        r_hat      = yhat_ln_t - ln_tm1                                # 預測對數報酬
    else:
        yhat_ln_t = np.nan; price_pred = np.nan; r_hat = np.nan

    # --- 模型建/續訓 ---
    if (model is None) or (MODE == 'refit'):
        model = build_small_lstm(WINDOW_DAY, N_FEATURES_THIS)  # 注意是 DAY 的窗口長度
        epochs_here = EPOCHS_INIT
    else:
        epochs_here = EPOCHS_STEP

    hist = model.fit(X_t, y_z_t, epochs=epochs_here, batch_size=1, shuffle=False, verbose=0)
    last_loss = float(hist.history['loss'][-1])

    # --- 真實值 ---
    p_tm1 = float(feat_df.loc[t_minus_1, 'ES1_CLOSE'])
    p_t   = float(feat_df.loc[t, 'ES1_CLOSE'])
    r_t   = float(feat_df.loc[t, 'ES1_LN_RET'])


    rows.append({
        'DATE'      : t,
        'ln_true'   : ln_t,
        'ln_pred'   : yhat_ln_t,
        'ret_true'  : r_t,
        'ret_pred'  : r_hat,
        'price_true': p_t,
        'price_pred': price_pred,
        'sigma_next': sigma_next,
        'train_loss': last_loss,
    })


# # ------------------ Evaluation ------------------

# ##########  風險
# # 1) 先建 df_out（此時 df_out 已存在 ret_pred、sigma_next）
# df_out = pd.DataFrame(rows).set_index("DATE").sort_index()

# # 2) 再一次性計算 VaR/ES（向量化）
# df_out[['VaR_ret', 'ES_ret', 'VaR_price', 'ES_price']] = compute_var_es(
#     mu=df_out['ret_pred'],
#     sigma=df_out['sigma_next'],
#     alpha=0.01,
#     nu=6,
#     price_base=feat_df.loc[df_out.index, 'ES1_CLOSE']  # 與 df_out.index 對齊
# )



############   回歸預測
df_eval = pd.DataFrame(rows).set_index('DATE').sort_index()

# 1) 濾掉第一筆（PRED_START 當下模型尚未建立），僅保留 PRED_START+1h ~ PRED_END
mask_eval = (df_eval.index > PRED_START) & (df_eval.index <= PRED_END)
df_pred_sw = df_eval.loc[mask_eval].copy()

# 2) 安全遮罩：僅評估有限實數
is_finite_ret  = np.isfinite(df_pred_sw['ret_true'])  & np.isfinite(df_pred_sw['ret_pred'])
is_finite_price= np.isfinite(df_pred_sw['price_true'])& np.isfinite(df_pred_sw['price_pred'])

# 3) Returns 指標
if is_finite_ret.any():
    mae_val  = float(mean_absolute_error(df_pred_sw.loc[is_finite_ret, 'ret_true'],
                                         df_pred_sw.loc[is_finite_ret, 'ret_pred']))
    rmse_val = float(np.sqrt(mean_squared_error(df_pred_sw.loc[is_finite_ret, 'ret_true'],
                                                df_pred_sw.loc[is_finite_ret, 'ret_pred'])))
    r2_ret   = (r2_score(df_pred_sw.loc[is_finite_ret, 'ret_true'],
                         df_pred_sw.loc[is_finite_ret, 'ret_pred'])
                if is_finite_ret.sum() > 1 else np.nan)

    mae_price  = float(mean_absolute_error(df_pred_sw.loc[is_finite_price, 'price_true'],
                                           df_pred_sw.loc[is_finite_price, 'price_pred']))
    rmse_price = float(np.sqrt(mean_squared_error(df_pred_sw.loc[is_finite_price, 'price_true'],
                                                  df_pred_sw.loc[is_finite_price, 'price_pred'])))
    r2_price   = (r2_score(df_pred_sw.loc[is_finite_price, 'price_true'],
                           df_pred_sw.loc[is_finite_price, 'price_pred'])
                  if is_finite_price.sum() > 1 else np.nan)

    # 方向性
    y_true_dir = (df_pred_sw.loc[is_finite_ret, 'ret_true'].values > 0).astype(int)
    y_pred_dir = (df_pred_sw.loc[is_finite_ret, 'ret_pred'].values > 0).astype(int)
    precision  = precision_score(y_true_dir, y_pred_dir, zero_division=0)
    recall     = recall_score(y_true_dir, y_pred_dir, zero_division=0)
    f1         = f1_score(y_true_dir, y_pred_dir, zero_division=0)

    # 簽策略（單純符號乘上真實報酬）
    strat = np.sign(df_pred_sw.loc[is_finite_ret, 'ret_pred']) * df_pred_sw.loc[is_finite_ret, 'ret_true']
    if len(strat) > 0:
        ann_factor  = 252  # 小時資料
        total_return = float(np.exp(strat.sum()) - 1)
        ann_return   = float(np.exp(strat.mean() * ann_factor) - 1)
        equity       = (1 + strat).cumprod()
        roll_max     = equity.cummax()
        max_dd       = float((equity / roll_max - 1).min())
    else:
        total_return = ann_return = max_dd = np.nan
else:
    mae_val = rmse_val = r2_ret = precision = recall = f1 = total_return = ann_return = max_dd = np.nan
    mae_price = rmse_price = r2_price = np.nan

# 4) Price 指標
if is_finite_price.sum() > 1:
    r2_price = r2_score(df_pred_sw.loc[is_finite_price, 'price_true'],
                        df_pred_sw.loc[is_finite_price, 'price_pred'])
else:
    r2_price = np.nan

# 5) 平均訓練損失
avg_loss = float(np.nanmean(df_pred_sw['train_loss'])) if len(df_pred_sw) else np.nan

# 6) 匯總表
metrics_df = pd.DataFrame({
    'asset':         [ASSET_SYMBOL_ES1],
    'group':         [GROUP_DAY],
    'start':         [TARGET_START_STR],
    'end':           [TARGET_END_STR],
    'mae':           [mae_val],
    'rmse':          [rmse_val],
    'R2_ret':        [r2_ret],
    'R2_price':      [r2_price],
    'mae_price': [mae_price],
    'rmse_price': [rmse_price],
    'DA':            [float(np.mean((df_pred_sw.loc[is_finite_ret,'ret_pred'] >= 0) ==
                                     (df_pred_sw.loc[is_finite_ret,'ret_true'] >= 0))) if is_finite_ret.any() else np.nan],
    'F1_Score':      [f1],
    'Precision':     [precision],
    'Recall':        [recall],
    'avg_loss':      [avg_loss],
    'total_return':  [total_return],
    'ann_return':    [ann_return],
    'max_drawdown':  [max_dd],
    'R2_ret':        [r2_ret],
    'R2_price':      [r2_price],
})

# ------------------ Save CSVs ------------------
stem = f"{ASSET_SYMBOL_ES1.lower()}_lstm_day_sliding_multifeat_nogarch"
os.makedirs("./LSTM_diagnostics", exist_ok=True)

out_csv = f"./LSTM_diagnostics/{stem}_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.csv"
df_pred_sw.to_csv(out_csv, float_format='%.10f')

metrics_vertical =  metrics_df.T.reset_index()
metrics_vertical.columns = ["metric", "value"]
metrics_csv = f"./LSTM_diagnostics/{stem}_metrics_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.csv"
metrics_vertical.to_csv(metrics_csv, index=False)

print("Saved:", out_csv)
print("Metrics:", metrics_csv)

# ------------------ Figures (3+1) ------------------
fig1 = f"./LSTM_diagnostics/{stem}_price_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig2 = f"./LSTM_diagnostics/{stem}_returns_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig3 = f"./LSTM_diagnostics/{stem}_scatter_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig4 = f"./LSTM_diagnostics/{stem}_trainloss_price_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"

plt.figure(figsize=(12,6))
plt.plot(df_pred_sw.index, df_pred_sw['price_true'], label='True Price')
plt.plot(df_pred_sw.index, df_pred_sw['price_pred'], label='Predicted Price', linestyle='--')
plt.xlabel('Time'); plt.ylabel('Price'); plt.title(f"{ASSET_SYMBOL_ES1} Price"); plt.legend(); plt.grid(True)
plt.savefig(fig1, dpi=300); plt.close()

plt.figure(figsize=(12,6))
plt.plot(df_pred_sw.index, df_pred_sw['ret_true'], label='True Return', alpha=0.7)
plt.plot(df_pred_sw.index, df_pred_sw['ret_pred'], label='Predicted Return', alpha=0.7)
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('Time'); plt.ylabel('Log Return'); plt.title(f"{ASSET_SYMBOL_ES1} Returns"); plt.legend(); plt.grid(True)
plt.savefig(fig2, dpi=300); plt.close()

plt.figure(figsize=(6,6))
plt.scatter(df_pred_sw['ret_true'], df_pred_sw['ret_pred'], alpha=0.5)
plt.axhline(0, color='black', linewidth=1); plt.axvline(0, color='black', linewidth=1)
plt.xlabel('True Return'); plt.ylabel('Predicted Return'); plt.title(f"{ASSET_SYMBOL_ES1} True vs Predicted Returns"); plt.grid(True)
plt.savefig(fig3, dpi=300); plt.close()

# Price + Train Loss
fig, ax1 = plt.subplots(figsize=(12,6))
ax1.plot(df_pred_sw.index, df_pred_sw['price_true'], label='True Price')
ax1.plot(df_pred_sw.index, df_pred_sw['price_pred'], label='Predicted Price', linestyle='--', alpha=0.9)
ax1.set_xlabel("Time"); ax1.set_ylabel("Price")
ax2 = ax1.twinx()
ax2.plot(df_pred_sw.index, df_pred_sw['train_loss'], label='Train Loss', alpha=0.6, color='green')
ax2.set_ylabel("Training Loss")
ax1.grid(True, which='both', axis='both', alpha=0.2)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout(); plt.savefig(fig4, dpi=300); plt.close()

print('Figures:', fig1, fig2, fig3, fig4)


<>:333: SyntaxWarning: invalid escape sequence '\h'
<>:333: SyntaxWarning: invalid escape sequence '\h'
/tmp/ipython-input-469698919.py:333: SyntaxWarning: invalid escape sequence '\h'
  mu : 預測條件均值（下一期對數報酬）  \hat{r}_{t+1}


Num GPUs: 1 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ memory growth set
Num GPUs: 1 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ memory growth set
是否可用GPU: True
使用中的裝置: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU 初始化流程完成
DatetimeIndex(['2004-03-29', '2004-03-30', '2004-03-31', '2004-04-01',
               '2004-04-02', '2004-04-05', '2004-04-06', '2004-04-07',
               '2004-04-08', '2004-04-12',
               ...
               '2025-06-17', '2025-06-18', '2025-06-20', '2025-06-23',
               '2025-06-24', '2025-06-25', '2025-06-26', '2025-06-27',
               '2025-06-30', '2025-07-01'],
              dtype='datetime64[ns]', name='DATE', length=5374, freq=None)
start 2005-05-31 00:00:00 
start 2005-06-01 00:00:00 
start 2005-06-02 00:00:00 
start 2005-06-03 00:00:00 
start 2005-06-06 00:00:00 
start 2005-06-07 00:00:00 
start 2005-06-08 00:00:00 
start 2005-06-09 00:00:00 
start 2005-06-10 0